## Demo of client side Union of hotset dataset as view over Kafka through ISK and coldset dataset on MiniIO

In [ ]:
# First shut down the packaged spark session

In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

25/10/15 16:15:21 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [2]:
spark.stop()

In [3]:
spark = None

In [ ]:
# Then reconnect with Spark Connect

In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()


spark

In [5]:
spark.sql("use hot.isk").show()
spark.sql("show tables").show()


++
||
++
++

+---------+----------------+-----------+
|namespace|       tableName|isTemporary|
+---------+----------------+-----------+
|      isk|transactions_old|      false|
|      isk|        accounts|      false|
|      isk|       customers|      false|
|      isk|        branches|      false|
|      isk|    transactions|      false|
+---------+----------------+-----------+





### ISK view over Kafka has multiple topics including two topics with transactions data:
    
 - **transactions_old** with static set of data of 500000 events sorted by TransactionTime - that can be loaded into the cold set on MiniIO for demo purposes
 - **transactions** with dynamic set of data - starting with 50 events and gets new event every second.



In [ ]:
# move some data into the coldset

In [ ]:
spark.sql("DROP TABLE IF EXISTS cold.data.customers")

In [7]:
spark.sql("""
CREATE TABLE cold.data.customers 
USING iceberg 
TBLPROPERTIES('format-version'='2') 
PARTITIONED BY (kafka_partition, truncate(1000, kafka_offset)) 
AS 
  SELECT * 
  FROM hot.isk.customers;
""")

DataFrame[]

In [33]:
spark.sql("SHOW TABLES IN cold.data").show()


+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|     data|           customers|      false|
|     data|index---customers...|      false|
|     data|streambased_metadata|      false|
+---------+--------------------+-----------+



In [8]:
spark.sql("SELECT COUNT(*) FROM cold.data.customers").show();

+--------+
|count(1)|
+--------+
|  400000|
+--------+



In [ ]:
# From here we should do it with hyperstream

In [57]:
import requests
import json
from IPython.display import JSON

In [58]:
req = { 'set': 'COLD' }
x = requests.post('http://hyperstream:9088/api/schema',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [59]:
req = { 
    'set': 'COLD',
    'index' : False,
    'sql' : 'SELECT * FROM customers LIMIT 10'
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [60]:
req = { 
    'set': 'COLD',
    'index' : False,
    'sql' : 'SELECT * FROM customers WHERE Name=\'Brendan Yost\''
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [61]:
req = { 
    'topic' : 'customers',
    'field' : 'Name'
      }
x = requests.put('http://hyperstream:9088/api/index',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [62]:
req = { 
    'topic' : 'customers',
    'field' : 'Name'
      }
x = requests.post('http://hyperstream:9088/api/index',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [64]:
req = { 
    'set': 'COLD',
    'index' : True,
    'sql' : 'SELECT * FROM customers WHERE Name=\'Brendan Yost\''
      }
x = requests.post('http://hyperstream:9088/api/query',json = req)
JSON(x.text)

<IPython.core.display.JSON object>

In [ ]:
# These are for investigating

In [8]:
spark.sql("DESCRIBE cold.data.customers").show()

+---------------+--------------------+-------+
|       col_name|           data_type|comment|
+---------------+--------------------+-------+
|     CustomerID|              string|   NULL|
|           Name|              string|   NULL|
|        Address|              string|   NULL|
|          Email|              string|   NULL|
|        PhoneNo|              string|   NULL|
|kafka_partition|                 int|   NULL|
|   kafka_offset|              bigint|   NULL|
|       kafka_ts|       timestamp_ntz|   NULL|
|               |                    |       |
| # Partitioning|                    |       |
|         Part 0|     kafka_partition|       |
|         Part 1|truncate(1000, ka...|       |
+---------------+--------------------+-------+



In [35]:
spark.sql("USE cold.data").show();

++
||
++
++



In [36]:
spark.sql("SELECT * FROM (SELECT *  FROM customers WHERE (( kafka_partition = 0 AND kafka_offset > 1000 AND kafka_offset < 2000)))  WHERE Name = 'Brendan Yost'").show()

+----------+----+-------+-----+-------+---------------+------------+--------+
|CustomerID|Name|Address|Email|PhoneNo|kafka_partition|kafka_offset|kafka_ts|
+----------+----+-------+-----+-------+---------------+------------+--------+
+----------+----+-------+-----+-------+---------------+------------+--------+



In [14]:
spark.sql("SELECT * FROM cold.data.`index---customers---Name` WHERE index_key = 'Brendan Yost' ORDER BY min_offset").show()

+------------+---------------+----------+----------+
|   index_key|kafka_partition|min_offset|max_offset|
+------------+---------------+----------+----------+
|Brendan Yost|              0|      1000|      2000|
+------------+---------------+----------+----------+



In [31]:
spark.sql("SELECT * FROM cold.data.streambased_metadata").show()

+---------+-----+---------------+----------+
|    topic|field|kafka_partition|max_offset|
+---------+-----+---------------+----------+
|customers| Name|              0|    399999|
+---------+-----+---------------+----------+



In [32]:
spark.sql("SELECT * FROM cold.data.`index---customers---Name` WHERE index_key = 'Brendan Yost' ORDER BY min_offset").show()

+------------+---------------+----------+----------+
|   index_key|kafka_partition|min_offset|max_offset|
+------------+---------------+----------+----------+
|Brendan Yost|              0|      1000|      2000|
+------------+---------------+----------+----------+



In [ ]:
import datetime

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers WHERE Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
WHERE c.Name = 'Brendan Yost' and c.kafka_offset >=0 and c.kafka_offset <=1000
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN cold.data.index_customers_name i
ON c.Name = i.Name AND c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())

In [ ]:
print( datetime.datetime.now())
spark.sql("""
SELECT * from cold.data.customers c
JOIN (SELECT * FROM cold.data.index_customers_name WHERE Name='Brendan Yost') i
ON c.kafka_partition = i.kafka_partition AND floor(c.kafka_offset/1000) = i.offset_batch
WHERE c.Name = 'Brendan Yost'
""").show()
print( datetime.datetime.now())